# HistGradientBoosting (HGBR) — Tanítás


Sklearn beépített **gradient boosting** regresszora. Lényegében ugyanaz mint a LightGBM/XGBoost, csak nem kell külön csomag. A `geostat_elemzes.py`-ban lévő paraméterezést követi (csapattárs hangolása), és **log-target transzformációval** tanul (a Duration jobbra ferde — `log1p` simítja). Ez egy **erősebb modell, mint a Decision Tree**, ezért a beadandóban a tetejére kerül.

Mentett kimenetek (Drive: `MODELS_DIR`):
- `hgbr_tuned.joblib` — a hangolt modell (TransformedTargetRegressor wrapper-rel együtt)
- `hgbr_predictions_test.npy` — test predikciók
- `hgbr_validation_curve_max_leaf_nodes.npz` — validation curve
- `hgbr_learning_curve.npz` — learning curve

Plus `results/metrics.csv` (Drive) — append-elt sorok.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Slityak/gepitan-seoul-bike-trip/blob/main/notebooks/05_hgbr.ipynb)

## 1. Setup (Colab + lokális kompatibilis)

In [ ]:
import os
import sys

IN_COLAB = 'google.colab' in sys.modules
BRANCH = 'main'

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')

    if not os.path.exists('/content/gepitan-beadando'):
        !git clone --branch {BRANCH} https://github.com/Slityak/gepitan-seoul-bike-trip.git /content/gepitan-beadando

    os.chdir('/content/gepitan-beadando')
    !git fetch origin {BRANCH}
    !git checkout {BRANCH}
    !git pull origin {BRANCH}
    !pip install -q -r requirements.txt
else:
    if os.path.basename(os.getcwd()) == 'notebooks':
        os.chdir('..')

sys.path.insert(0, '.')
print(f'Working dir: {os.getcwd()}')
print(f'In Colab: {IN_COLAB}')
print(f'Branch: {BRANCH}')

In [ ]:
import numpy as np
import joblib
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.compose import TransformedTargetRegressor
from sklearn.model_selection import GridSearchCV, validation_curve, learning_curve

from src.config import RANDOM_SEED, CV_FOLDS, MODELS_DIR, ensure_dirs
from src.data_io import load_v1_data
from src.evaluation import append_metrics, evaluate_model

ensure_dirs()
MODELS_DIR.mkdir(parents=True, exist_ok=True)

## 2. Adatbetöltés

In [ ]:
X_train, X_test, y_train, y_test, feature_names = load_v1_data()

print(f'Train shape: {X_train.shape}, Test shape: {X_test.shape}')
print(f'Target — train: mean={y_train.mean():.2f}, std={y_train.std():.2f}, '
      f'min={y_train.min()}, max={y_train.max()}')

## 3. Baseline: alap HGBR (no tuning, no log-target)

Default `HistGradientBoostingRegressor` — referenciapont, hogy lássuk mit ad a hangolás és a log-target transzformáció.

In [ ]:
hgbr_default = HistGradientBoostingRegressor(random_state=RANDOM_SEED)

result_default, _ = evaluate_model(
    hgbr_default,
    X_train, y_train, X_test, y_test,
    model_name='HGBR (default)',
    notes='no tuning, no log-target',
)

print(f'MAE: {result_default.mae:.3f}')
print(f'RMSE: {result_default.rmse:.3f}')
print(f'R²: {result_default.r2:.3f}')
print(f'Train idő: {result_default.train_time_sec:.2f} s')

append_metrics(result_default)

## 4. GridSearchCV hangolás (sample-en) + log-target wrapper + refit a teljes train-en

1. `TUNE_SAMPLE_SIZE` random sample a train-ből → ezen GridSearchCV (a hangolás mindig a nyers HGBR-en történik)
2. A legjobb paraméterekből építünk egy **`TransformedTargetRegressor`** wrapper-t, ami `log1p`/`expm1` transzformációt alkalmaz a target-en (Duration jobbra ferde, ezért log-térben tanulni stabilabb)
3. Refit a teljes train-en
4. Mentés: modell + test predikciók + metrika

In [ ]:
TUNE_SAMPLE_SIZE = 500_000

param_grid = {
    'max_iter': [300, 500, 700],
    'learning_rate': [0.05, 0.1],
    'max_leaf_nodes': [31, 63],
    'min_samples_leaf': [30],
    'l2_regularization': [0.0, 0.05],
}

n_tune = min(TUNE_SAMPLE_SIZE, len(X_train))
rng = np.random.default_rng(RANDOM_SEED)
tune_idx = rng.choice(len(X_train), size=n_tune, replace=False)
X_tune, y_tune = X_train[tune_idx], y_train[tune_idx]

grid = GridSearchCV(
    estimator=HistGradientBoostingRegressor(
        early_stopping=True,
        validation_fraction=0.1,
        n_iter_no_change=30,
        random_state=RANDOM_SEED,
    ),
    param_grid=param_grid,
    cv=CV_FOLDS,
    scoring='neg_mean_absolute_error',
    n_jobs=-1,
    verbose=1,
)
grid.fit(X_tune, y_tune)

print(f'\nTune sample size: {n_tune}')
print(f'Legjobb paraméterek: {grid.best_params_}')
print(f'CV legjobb score (neg MAE): {grid.best_score_:.3f}')

best_hgbr = HistGradientBoostingRegressor(
    **grid.best_params_,
    early_stopping=True,
    validation_fraction=0.1,
    n_iter_no_change=30,
    random_state=RANDOM_SEED,
)
best_model = TransformedTargetRegressor(
    regressor=best_hgbr,
    func=np.log1p,
    inverse_func=np.expm1,
)

result_tuned, y_pred_tuned = evaluate_model(
    best_model,
    X_train, y_train, X_test, y_test,
    model_name='HGBR (tuned + log-target)',
    notes=f'GridSearchCV cv={CV_FOLDS} on n={n_tune}, log1p/expm1 target wrapper, refit on full train',
)

print(f'\nTeszt MAE: {result_tuned.mae:.3f}')
print(f'Teszt RMSE: {result_tuned.rmse:.3f}')
print(f'Teszt R²: {result_tuned.r2:.3f}')

append_metrics(result_tuned)

joblib.dump(best_model, MODELS_DIR / 'hgbr_tuned.joblib')
np.save(MODELS_DIR / 'hgbr_predictions_test.npy', y_pred_tuned)
print(f'\nMentve: {MODELS_DIR}/hgbr_tuned.joblib + hgbr_predictions_test.npy')

## 5. Validation curve (max_leaf_nodes)

Megmutatja, hogy a `max_leaf_nodes` változtatása hogyan hat a CV teljesítményre — analóg a Decision Tree `max_depth` görbéjével, de a HGBR-nél a fa-méretet a leaf-szám szabályozza.

In [ ]:
DIAG_SAMPLE_SIZE = 50_000

diag_rng = np.random.default_rng(RANDOM_SEED)
diag_idx = diag_rng.choice(len(X_train), size=min(DIAG_SAMPLE_SIZE, len(X_train)), replace=False)
X_diag, y_diag = X_train[diag_idx], y_train[diag_idx]
print(f'Diagnosztikai sample: {X_diag.shape}')

vc_param_range = [7, 15, 31, 63, 127]
vc_train_scores, vc_test_scores = validation_curve(
    HistGradientBoostingRegressor(
        max_iter=300,
        learning_rate=0.1,
        random_state=RANDOM_SEED,
    ),
    X_diag, y_diag,
    param_name='max_leaf_nodes',
    param_range=vc_param_range,
    cv=CV_FOLDS,
    scoring='neg_mean_absolute_error',
    n_jobs=-1,
)
vc_train_mae = -vc_train_scores
vc_test_mae = -vc_test_scores

vc_labels = np.array([str(p) for p in vc_param_range])
np.savez(
    MODELS_DIR / 'hgbr_validation_curve_max_leaf_nodes.npz',
    param_range_labels=vc_labels,
    train_mae=vc_train_mae,
    test_mae=vc_test_mae,
)
print(f'Validation curve mentve: {MODELS_DIR}/hgbr_validation_curve_max_leaf_nodes.npz')
for lbl, tr, te in zip(vc_labels, vc_train_mae.mean(axis=1), vc_test_mae.mean(axis=1)):
    print(f'  max_leaf_nodes={lbl:>4s}: train MAE={tr:.3f}, CV MAE={te:.3f}')

## 6. Learning curve

Megmutatja, hogy a tuned modell mennyi adattal éri el a plateaut — analóg a Decision Tree-nél látott görbével.

In [ ]:
lc_train_sizes = np.linspace(0.1, 1.0, 8)
lc_sizes, lc_train_scores, lc_test_scores = learning_curve(
    HistGradientBoostingRegressor(
        **grid.best_params_,
        early_stopping=True,
        validation_fraction=0.1,
        n_iter_no_change=30,
        random_state=RANDOM_SEED,
    ),
    X_diag, y_diag,
    train_sizes=lc_train_sizes,
    cv=CV_FOLDS,
    scoring='neg_mean_absolute_error',
    n_jobs=-1,
    shuffle=True,
    random_state=RANDOM_SEED,
)
lc_train_mae = -lc_train_scores
lc_test_mae = -lc_test_scores

np.savez(
    MODELS_DIR / 'hgbr_learning_curve.npz',
    sizes=lc_sizes,
    train_mae=lc_train_mae,
    test_mae=lc_test_mae,
)
print(f'Learning curve mentve: {MODELS_DIR}/hgbr_learning_curve.npz')
for sz, tr, te in zip(lc_sizes, lc_train_mae.mean(axis=1), lc_test_mae.mean(axis=1)):
    print(f'  size={int(sz):>6d}: train MAE={tr:.3f}, CV MAE={te:.3f}')